# Climate Crop Yield Intelligence 🌾🌍

I'M data scientist engineer and one of my clients is an agriculture company that wants to understand how climate pressure relates to crop yield.

The client does not only want charts. They want to know **where risk looks higher, which crops deserve attention, and whether climate data actually improves forecasting.**

This is a full 2026 rebuild of a university idea first explored around 2022.

## Problem Statement

I will analyze temperature, precipitation and farm-management conditions across six crops and ask practical business questions.

**IMPORTANT NOTE:** association is not causation. I will use the results as signals, not proof.

## 1. Importing Libraries

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from climate_crop_yield.analysis import build_risk_table, yield_change_summary
from climate_crop_yield.data import build_analysis_panel
from climate_crop_yield.features import add_detrended_residuals, add_features, add_temperature_bins, crop_temperature_sensitivity, validate_panel
from climate_crop_yield.model import fit_time_split
from climate_crop_yield.plots import label_bars, set_project_style

set_project_style()
pd.set_option("display.max_columns", 30)

## 2. Dataset Overview

V1 uses public data: FAO crop yields, ERA5 temperature/precipitation, and FAO/World Bank fertilizer and irrigation indicators through reproducible OWID Grapher CSV endpoints.

**Crops:** Wheat · Maize · Rice · Potatoes · Soybeans · Barley  
**Period:** 1990–2023

In [ ]:
df = build_analysis_panel(1990, 2023)
validate_panel(df)
df = add_features(df)

print(f"Rows: {len(df):,}")
print("Countries:", df["Code"].nunique())
print("Crops:", df["crop"].nunique())
print("Years:", df["Year"].min(), "-", df["Year"].max())

pd.DataFrame({
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
}).sort_values("missing_pct", ascending=False)

**Data decision:** irrigation has only about 23% coverage in the live panel. I keep it for exploratory analysis but exclude it from the core predictive model instead of mostly imputing it.

## 3. Business Questions

### 1. Which crops improved the most since 1990?

In [ ]:
trend = df.groupby(["crop","Year"], as_index=False)["yield_t_ha"].median()
sns.lineplot(data=trend, x="Year", y="yield_t_ha", hue="crop", palette="Set2", linewidth=2)
plt.title("Median Crop Yield Across Countries (1990–2023)")
plt.ylabel("Yield (t/ha)")
plt.tight_layout()
plt.show()

yield_change_summary(df).round(2)

### 2. What happens in warmer-than-trend years?

Raw temperature and raw yield both have long-term trends. I remove the linear time trend **inside every country × crop history** before comparing them.

In [ ]:
d = add_detrended_residuals(df).dropna(subset=["temp_detrended_c","yield_detrended_t_ha"])
plot_d = d.sample(min(12000, len(d)), random_state=42)

g = sns.lmplot(
    data=plot_d, x="temp_detrended_c", y="yield_detrended_t_ha",
    col="crop", col_wrap=3, height=3.3, aspect=1.2,
    scatter_kws={"alpha":0.16,"s":14},
    line_kws={"color":"green","linewidth":2},
)
g.set_axis_labels("Detrended temperature residual (°C)", "Detrended yield residual (t/ha)")
g.fig.suptitle("Interannual Temperature vs Yield After Removing Time Trends", y=1.03)
plt.show()

### 3. Which crops look most temperature sensitive?

In [ ]:
sensitivity = crop_temperature_sensitivity(df)

colors = ["#f7786b" if x < 0 else "#99ff99" for x in sensitivity["yield_change_t_ha_per_1c_deviation"]]
ax = sns.barplot(data=sensitivity, x="crop", y="yield_change_t_ha_per_1c_deviation",
                 hue="crop", palette=colors, legend=False)
plt.axhline(0, color="black", linewidth=1)
plt.title("Detrended Yield Association per +1°C")
plt.ylabel("Yield association (t/ha per +1°C)")
label_bars(ax, 3)
plt.tight_layout()
plt.show()

sensitivity.round(4)

### 4. Is there one perfect temperature?

No magic exact number this time. I use broad ranges only as an exploratory view; they mix geography and production systems, so I do **not** call any bin a physiological optimum.

In [ ]:
binned = add_temperature_bins(df.dropna(subset=["temperature_c","yield_t_ha"]), 8)
temp = binned.groupby(["crop","temp_bin"], observed=True)["yield_t_ha"].median().reset_index()
g = sns.catplot(data=temp, x="temp_bin", y="yield_t_ha", col="crop", col_wrap=2,
                kind="bar", palette="Greens", sharex=False, sharey=False, height=3.5, aspect=1.4)
g.set_xticklabels(rotation=55, ha="right")
g.set_axis_labels("Observed temperature range", "Median yield (t/ha)")
g.fig.suptitle("Temperature Ranges — Exploratory, Not a Magic Degree", y=1.02)
plt.show()

### 5. Does more rain always mean better yield?

In [ ]:
rain = df.dropna(subset=["precip_deviation_mm","yield_yoy_pct_w"])
rain = rain.sample(min(12000, len(rain)), random_state=42)
g = sns.lmplot(data=rain, x="precip_deviation_mm", y="yield_yoy_pct_w",
               col="crop", col_wrap=3, height=3.3, aspect=1.2,
               scatter_kws={"alpha":0.14,"s":13}, line_kws={"color":"#66b3ff"})
g.set_axis_labels("Precipitation deviation from country mean (mm)", "YoY yield change (%)")
plt.show()

### 6. What do irrigation and fertilizer tell us?

Both are observational country-level indicators. Irrigation is shown only on its observed subset because coverage is limited.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, col, title in [
    (axes[0],"irrigated_land_pct","Irrigation"),
    (axes[1],"fertilizer_kg_ha","Fertilizer"),
]:
    x = df.dropna(subset=[col,"yield_t_ha"]).copy()
    x["group"] = pd.qcut(x[col], 4, labels=["Low","Mid-low","Mid-high","High"], duplicates="drop")
    sns.boxplot(data=x, x="group", y="yield_t_ha", hue="group", palette="Set2",
                legend=False, showfliers=False, ax=ax)
    ax.set_title(f"Yield by {title} Intensity — Descriptive")
    ax.set_xlabel(f"{title} group")
    ax.set_ylabel("Yield (t/ha)")
plt.tight_layout()
plt.show()

print(f"Irrigation coverage: {100*df['irrigated_land_pct'].notna().mean():.2f}%")

### 7. Where is climate risk highest?

The V1 screening score combines historical yield volatility with a negative **detrended** temperature-yield association. It prioritizes investigation; it is not an insurance-grade probability.

In [ ]:
risk = build_risk_table(df)
risk[["Entity","crop","risk_score","volatility_cv","temp_slope","observations"]].head(15).round(4)

### 8. Can climate information beat strong forecasting baselines?

Train: **before 2018** · Test: **2018 onward**

The model predicts a residual correction using train-only temperature, precipitation and fertilizer anomalies. I compare it with crop median, country×crop median, and persistence (last pre-2018 yield). If the simple baseline wins, I keep the result.

In [ ]:
result = fit_time_split(df, split_year=2018, random_state=42)
pd.Series(result.metrics, name="value").to_frame().round(4)

In [ ]:
pred = result.predictions
p = pred.sample(min(4000,len(pred)), random_state=42)
sns.scatterplot(data=p, x="yield_t_ha", y="predicted_yield_t_ha", hue="crop", palette="Set2", alpha=.6)
lim = max(pred["yield_t_ha"].max(), pred["predicted_yield_t_ha"].max())
plt.plot([0,lim],[0,lim],"--",color="black",linewidth=1)
plt.title("Actual vs Predicted Yield — 2018+ Holdout")
plt.tight_layout()
plt.show()

pred.groupby("crop")["abs_error"].agg(["mean","median","count"]).sort_values("mean", ascending=False).round(4)

## 4. Final Business Takeaways

- **24,892** country-year-crop observations, **187** countries/territories, six crops, 1990–2023.
- Maize has the largest median yield increase: **+121.6%** comparing 1990–1994 with 2019–2023.
- After detrending, all six pooled temperature-yield slopes are negative; Potatoes are strongest at **-0.2725 t/ha per +1°C**, followed by Maize at **-0.1715**.
- V1's risk screen starts with **Cameroon–Potatoes**, **Oman–Barley**, and **Oman–Maize**.
- Model: **MAE 1.4473 t/ha**, **R² 0.9038** on 2018+.
- It beats the static country×crop median by **10.62%**, but persistence is stronger at **0.8976 MAE**.
- For short-horizon forecasting, **recent production history beats this annual climate-anomaly correction**.

## 5. Limitations

Country averages hide local farms; annual climate hides heat waves and rainfall timing; irrigation coverage is limited; management variables are observational; detrending reduces time-trend confounding but does not establish causality.

Growing-season-specific climate and crop calendars are the clearest next data upgrade.

## Conclusion

This rebuild keeps the original business idea but upgrades the evidence: real public data, clean country filtering, detrended climate analysis, leakage-safe prediction, strong baselines and explicit failures.

**Final client message:** climate risk cannot be removed, but it can be measured more carefully — and sometimes the most useful result is learning what does *not* beat a simple baseline.